# ⚙️ vLLM 调度器 — Continuous Batching 深度解析

**本文目标**：深入理解 vLLM Scheduler 的调度算法、Continuous Batching 的实现细节和抢占策略。

读完这篇你会理解：
- Scheduler 的状态机和三个队列
- Continuous Batching 的 batch 构建逻辑
- Preemption (抢占) 的触发条件和策略
- 调度策略如何影响 TTFT 和吞吐

## 1. Scheduler 的状态机

```
                  ┌──────────┐
         到达     │ WAITING  │
       ─────────→ │          │
                  └────┬─────┘
                       │ _schedule() 检查条件通过
                       ▼
                  ┌──────────┐
                  │ RUNNING  │ ←──── 被抢占后重新调度 ────┐
                  │          │                             │
                  └────┬─────┘                             │
                       │                                   │
          ┌────────────┼────────────┐                      │
          │            │            │                      │
     生成结束    显存紧张       用户取消                    │
          │     (OOM 风险)        │                        │
          ▼            ▼            ▼                      │
      ┌──────┐   ┌──────────┐  ┌──────┐                   │
      │ DONE │   │ SWAPPED  │  │ DONE │                   │
      └──────┘   │ (KV Cache│  └──────┘                   │
                 │  → CPU)  │                              │
                 └────┬─────┘                              │
                      │ 显存缓解后                          │
                      └────────────────────────────────────┘
```

### 1.1 三个队列

```python
class Scheduler:
    waiting: List[SequenceGroup]   # 等待处理的请求
    running: List[SequenceGroup]   # 正在生成 token 的请求
    swapped: List[SequenceGroup]   # 被抢占 (KV Cache 换出到 CPU) 的请求
    
    # 调度策略
    policy: str = "fcfs"  # 默认先到先服务
    # 也支持 "priority" (按优先级)
```

### 1.2 _schedule() 的核心逻辑

```python
def _schedule(self) -> SchedulerOutput:
    """每次 forward 前调用, 返回本次 step 的调度结果"""
    
    # Step 1: 检查 running 请求的状态
    # - 哪些请求完成了? → 移到 DONE, 释放 KV Cache
    # - 哪些请求需要被抢占? → 移到 SWAPPED
    
    # Step 2: 尝试从 waiting/swapped 队列拉请求
    # - 检查显存: 有新请求的 KV Cache 空间吗? (问 Block Manager)
    # - 检查 batch 容量: 当前 batch < max_num_seqs?
    # - 如果条件满足 → 分配 KV Cache blocks → 加入 running
    
    # Step 3: 构建本 step 的 batch
    # - 收集所有 running 请求的 sequence
    # - 构建 input tensors (token IDs, positions, block tables...)
    # - 返回 SchedulerOutput
    
    scheduled_seqs = []
    preempted_seqs = []
    
    # ... 调度逻辑 ...
    
    return SchedulerOutput(
        scheduled_seq_groups=scheduled_seqs,
        preempted_seq_groups=preempted_seqs,
        blocks_to_swap_in=...,
        blocks_to_swap_out=...,
        blocks_to_copy=...,
    )
```

## 2. Continuous Batching 的批次构建

### 2.1 关键: Prefill 和 Decode 混合

```python
# vLLM 一次 forward 可以同时包含:
# - 一个 prefill 请求 (处理整个 prompt)
# - 多个 decode 请求 (各自生成一个 token)

# 但只能有一个 prefill 请求! (因为 prefill 的计算量远大于 decode)
# 如果多个请求同时需要 prefill → 排队, 每次取一个

# 批次构建示例:
# 当前 running: [seq_A (decode 第 87 步), seq_B (decode 第 12 步)]
# waiting:      [seq_C (prefill 500 tokens), seq_D (prefill 100 tokens)]
#
# Scheduler 决定: 本次 batch = seq_C 的 prefill + seq_A/B 的 decode
# → [prefill: seq_C(500)] [decode: seq_A(1)] [decode: seq_B(1)]
```

### 2.2 Batch Token Budget

vLLM 引入 `max_num_batched_tokens` 来控制每个 batch 的总 token 数:

```bash
# 配置
--max-num-batched-tokens 8192  # 每个 batch 最多 8192 个 token

# 作用: 防止单个 batch 太大导致 OOM 或延迟过高
# 示例:
#   seq_A (prefill 5000 tokens): 符合 budget (5000 < 8192) ✓
#   seq_B (prefill 10000 tokens): 拆分或推迟 ✗
#   seq_C (decode 1 token): 总是符合 ✓

# 对混合推理的影响:
#   Agent 场景: tool result 注入 → 可能大 prefill
#   → 超过 budget → 被拆分 → 增加 TTFT
#   建议: Agent 场景设大一点, 如 --max-num-batched-tokens 16384
```

## 3. Preemption (抢占) 策略

### 3.1 什么时候触发抢占？

```python
# 场景: 显存紧张
# 当前 KV Cache 使用率: 92%
# 新请求到达, 需要 2 GB KV Cache → 不够!

# vLLM 的响应:
# 1. 检查是否有 "可以安全抢占" 的请求
#    - 优先级最低的 running 请求
#    - 或者是 waiting 中的请求 (直接拒绝, 不抢占)
# 2. 选择一个/多个 running 请求 → 抢占
# 3. 将其 KV Cache blocks 复制到 CPU 内存 (swap out)
# 4. 释放 GPU 显存 → 分配给新请求
# 5. 被抢占的请求进入 SWAPPED 队列
# 6. 当 GPU 显存缓解 → swap in → 恢复运行
```

### 3.2 Preemption Mode 配置

```bash
# vLLM 的抢占模式
--preemption-mode recompute  # 默认: 丢弃 KV Cache, 重新 prefill
--preemption-mode swap       # 换出到 CPU, 恢复时从断点继续

# Recompute (推荐):
#   丢弃被抢占请求的 KV Cache
#   恢复时: 用已有的 token_ids 重新 prefill
#   优点: 不需要 CPU 显存
#   缺点: 恢复时重新计算 → 增加一些延迟

# Swap:
#   把 KV Cache 复制到 CPU 内存
#   恢复时: swap in + 继续 decode
#   优点: 恢复快 (不需要重算)
#   缺点: 需要大量 CPU 内存, 且 swap 时间不可忽略
```

## 4. 调度策略对比

| 策略 | 行为 | 最适合 |
|------|------|--------|
| **FCFS** (默认) | 先到先服务 | 通用场景 |
| **Priority** | 优先级高的先调度 | 混合负载 (短请求优先) |
| **LPM** (Longest Prefix Match) | 最长公共前缀的先调度 | Agent / 前缀较多 |

```python
# vLLM 调度器的可插拔设计
# 可以自定义调度策略:

from vllm.core.scheduler import Scheduler

class PriorityScheduler(Scheduler):
    """优先调度短请求 (降低 P50 延迟)"""
    def _schedule(self):
        # 估算每个 waiting 请求的 prefill 时间
        # 短的优先进 batch
        self.waiting.sort(key=lambda sg: sg.get_seqs()[0].get_len())
        return super()._schedule()
```

## 5. 代码实验: 调度模拟器

In [ ]:
# vLLM Scheduler 简化模拟

import random
from collections import deque

class SimScheduler:
    def __init__(self, max_seqs=8, max_tokens_per_batch=8192, block_size=16):
        self.max_seqs = max_seqs
        self.max_tokens_per_batch = max_tokens_per_batch
        self.block_size = block_size
        self.waiting = deque()
        self.running = []
        self.total_blocks = 1000
        self.free_blocks = self.total_blocks
        self.stats = {"preemptions": 0, "completed": 0, "total_steps": 0}
    
    def add_request(self, req_id, prompt_len, max_tokens):
        n_blocks = (prompt_len + max_tokens + self.block_size - 1) // self.block_size
        r = {"id": req_id, "prompt_len": prompt_len, "max_tokens": max_tokens,
             "tokens_generated": 0, "blocks_reserved": n_blocks, "prefilled": False}
        self.waiting.append(r)
    
    def step(self):
        self.stats["total_steps"] += 1
        
        # 1. 清理完成的请求
        for r in self.running[:]:
            if r["tokens_generated"] >= r["max_tokens"]:
                self.running.remove(r)
                self.free_blocks += r["blocks_reserved"]
                self.stats["completed"] += 1
        
        # 2. 尝试从 waiting 拉请求
        while self.waiting and len(self.running) < self.max_seqs:
            r = self.waiting[0]
            if self.free_blocks >= r["blocks_reserved"]:
                self.waiting.popleft()
                self.running.append(r)
                self.free_blocks -= r["blocks_reserved"]
            else:
                # 显存不够 → 抢占!
                if self.running:
                    victim = max(self.running, key=lambda x: x["blocks_reserved"])
                    print(f"    [抢占] Seq {victim['id']} (reserved {victim['blocks_reserved']} blocks)")
                    self.running.remove(victim)
                    self.free_blocks += victim["blocks_reserved"]
                    self.stats["preemptions"] += 1
                else:
                    break  # 真的 OOM
        
        # 3. 执行一步 decode
        for r in self.running[:]:
            if not r["prefilled"]:
                r["prefilled"] = True  # prefill 完成
            r["tokens_generated"] += 1  # 生成一个 token
    
    def run(self, max_steps=100):
        for _ in range(max_steps):
            if not self.waiting and not self.running:
                break
            self.step()

# === 测试 ===
print("=" * 60)
print("vLLM Scheduler 模拟")
print("=" * 60)

random.seed(123)
sim = SimScheduler(max_seqs=8, max_tokens_per_batch=8192)

# 提交 30 个请求 (不同的 prompt 长度和 max_tokens)
for i in range(30):
    sim.add_request(i,
        prompt_len=random.choice([50, 200, 500, 2000, 5000]),
        max_tokens=random.randint(50, 500))

print(f"请求数: 30")
print(f"Max concurrent: 8")
print()
sim.run(max_steps=200)

print(f"\n结果:")
print(f"  完成: {sim.stats['completed']}/{30}")
print(f"  总 steps: {sim.stats['total_steps']}")
print(f"  抢占次数: {sim.stats['preemptions']}")
print(f"  平均每个请求: {sim.stats['total_steps']/sim.stats['completed']:.1f} steps")

# 对比: 纯文本 vs Agent (长 prompt) 的延迟
print(f"\n模拟: 混合负载")
sim2 = SimScheduler(max_seqs=8)
for i in range(20):
    # 10 个短 prompt (纯文本)
    sim2.add_request(i, prompt_len=100, max_tokens=100)
    # 10 个长 prompt (Agent)
    sim2.add_request(20+i, prompt_len=3000, max_tokens=300)

print(f"  短请求: 10 个 (prompt=100, max=100)")
print(f"  长请求: 10 个 (prompt=3000, max=300)")
sim2.run(max_steps=300)
print(f"  完成: {sim2.stats['completed']}/20")
print(f"  抢占: {sim2.stats['preemptions']}")
print(f"  观察: 长 prefill 的 Agent 请求会延迟短请求的 prefill")